# AEGIS M1D: DarijaBERT-mix Fine-Tuning

**Goal**: Fine-tune `SI2M-Lab/DarijaBERT-mix` as M1D - a binary offensive language detector for Moroccan Darija.

**Architecture mirror**: Darija equivalent of your English M1 (XLM-R gatekeeper). Same binary task, same cascading logic, but Darija-native.

**Dataset**: Merged OMCD (Mendeley V4 + GitHub) - 24K train / 4.3K val / 1.2K test

**Expected runtime**: ~30-50 min on Colab T4 (free tier)

## 1. Setup & Dependencies

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn matplotlib seaborn

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    print(f'VRAM: {props.total_memory / 1e9:.1f} GB')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 150, 'font.size': 11})

DATA_DIR = '/content/drive/MyDrive/AEGIS_ML/data/darija/merged_dataset'
OUTPUT_DIR = '/content/drive/MyDrive/AEGIS_ML/models/M1D_darijabert'
PLOTS_DIR = os.path.join(OUTPUT_DIR, 'plots')
os.makedirs(PLOTS_DIR, exist_ok=True)
print(f'Data dir: {os.path.exists(DATA_DIR)}, Output: {OUTPUT_DIR}')

## 2. Load Dataset

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_merged.csv'))
val_df = pd.read_csv(os.path.join(DATA_DIR, 'val_merged.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test_heldout.csv'))

for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f'{name}: {len(df):,}  (offensive: {df["label"].sum():,}, ratio: {df["label"].mean():.3f})')

print('\n--- Offensive samples ---')
for _, r in train_df[train_df['label']==1].sample(3, random_state=42).iterrows():
    print(f'  {r["text"][:120]}')
print('\n--- Non-offensive samples ---')
for _, r in train_df[train_df['label']==0].sample(3, random_state=42).iterrows():
    print(f'  {r["text"][:120]}')

### Plot: Dataset Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
splits = ['Train', 'Validation', 'Test']
off = [train_df['label'].sum(), val_df['label'].sum(), test_df['label'].sum()]
nonoff = [len(train_df)-off[0], len(val_df)-off[1], len(test_df)-off[2]]
x = np.arange(len(splits)); w = 0.35
axes[0].bar(x-w/2, nonoff, w, label='Non-Offensive', color='#4CAF50')
axes[0].bar(x+w/2, off, w, label='Offensive', color='#F44336')
axes[0].set_ylabel('Count'); axes[0].set_title('Class Distribution by Split')
axes[0].set_xticks(x); axes[0].set_xticklabels(splits); axes[0].legend()
for i,(n,o) in enumerate(zip(nonoff,off)):
    axes[0].text(i-w/2,n+100,f'{n:,}',ha='center',fontsize=9)
    axes[0].text(i+w/2,o+100,f'{o:,}',ha='center',fontsize=9)

ratios = [df['label'].mean() for df in [train_df, val_df, test_df]]
bars = axes[1].bar(splits, ratios, color=['#1976D2','#FF9800','#9C27B0'], width=0.5)
axes[1].axhline(y=0.5, color='gray', ls='--', alpha=0.5, label='Perfect balance')
axes[1].set_ylabel('Offensive Ratio'); axes[1].set_title('Class Balance'); axes[1].set_ylim(0,0.7); axes[1].legend()
for b,r in zip(bars,ratios):
    axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'{r:.3f}', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'dataset_distribution.png'), bbox_inches='tight')
plt.show()

## 3. Darija Text Preprocessing

- Normalize Arabic character variants (Alif, Ya)
- Strip diacritics (tashkeel) and tatweel
- Keep emojis (DarijaBERT handles them)
- Optional Arabizi digit normalization

In [ ]:
ALIF_VARIANTS = ['\u0623','\u0625','\u0622','\u0671']
YA_VARIANTS = ['\u0649']
TASHKEEL_RE = re.compile(r'[\u064B-\u065F\u0670]')
TATWEEL_RE = re.compile(r'\u0640')
ARABIZI_MAP = str.maketrans({'7':'\u062D','9':'\u0642','3':'\u0639','5':'\u062E','2':'\u0627','8':'\u0642'})
ARABIZI_DG = [('ch','\u0634'),('sh','\u0634'),('gh','\u063A'),('kh','\u062E'),('th','\u062B'),('dh','\u0630')]

def normalize_darija(text, apply_arabizi=False):
    if not text or not isinstance(text, str): return ''
    text = text.strip()
    for v in ALIF_VARIANTS: text = text.replace(v, '\u0627')
    for v in YA_VARIANTS: text = text.replace(v, '\u064A')
    text = TASHKEEL_RE.sub('', text)
    text = TATWEEL_RE.sub('', text)
    if apply_arabizi:
        for dg,ar in ARABIZI_DG: text = text.replace(dg, ar)
        text = text.translate(ARABIZI_MAP)
    return re.sub(r'\s+',' ',text).strip()

tests = [('\u0623\u0646\u0627 \u063A\u0627\u062F\u064A \u0646\u0642\u062A\u0644\u0643','threat'),
         ('nta 7mar ghadi ndarbek','Arabizi threat'),
         ('sir t9awed a wld l9a7ba','Arabizi insult'),
         ('\u0633\u0644\u0627\u0645 \u062E\u0648\u064A\u0627 \u0644\u0627\u0628\u0627\u0633','safe greeting')]
print('=== Normalizer Test ===')
for t,d in tests:
    print(f'{d}: {t} -> {normalize_darija(t)}')

In [ ]:
# Apply normalization (no Arabizi - DarijaBERT-mix handles both scripts natively)
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    df['text_norm'] = df['text'].apply(lambda x: normalize_darija(x))
    empty = df['text_norm'].str.len() < 2
    if empty.any():
        print(f'{name}: dropped {empty.sum()} empty rows')
        df.drop(df[empty].index, inplace=True); df.reset_index(drop=True, inplace=True)
    print(f'{name}: {len(df):,} rows')

## 4. Tokenization

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = 'SI2M-Lab/DarijaBERT-mix'
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Vocab: {tokenizer.vocab_size:,}, Max len: {tokenizer.model_max_length}')

def tok(ex): return tokenizer(ex['text_norm'], truncation=True, max_length=MAX_LENGTH, padding='max_length')

train_ds = Dataset.from_pandas(train_df[['text_norm','label']]).map(tok, batched=True)
val_ds   = Dataset.from_pandas(val_df[['text_norm','label']]).map(tok, batched=True)
test_ds  = Dataset.from_pandas(test_df[['text_norm','label']]).map(tok, batched=True)
print(f'Tokenized: train={len(train_ds):,} val={len(val_ds):,} test={len(test_ds):,}')

## 5. Training

Hyperparameters: LR=2e-5, warmup=10%, batch=16, epochs=4, weight_decay=0.01

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

def compute_metrics(ep):
    logits, labels = ep
    preds = np.argmax(logits, axis=-1)
    return {'f1_macro': f1_score(labels,preds,average='macro'),
            'f1_weighted': f1_score(labels,preds,average='weighted'),
            'accuracy': accuracy_score(labels,preds)}

In [ ]:
BATCH_SIZE, NUM_EPOCHS, LEARNING_RATE = 16, 4, 2e-5
WARMUP_RATIO, WEIGHT_DECAY = 0.1, 0.01

args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=32,
    learning_rate=LEARNING_RATE, warmup_ratio=WARMUP_RATIO, weight_decay=WEIGHT_DECAY,
    logging_steps=50, eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='f1_macro', greater_is_better=True,
    fp16=torch.cuda.is_available(), report_to='none', save_total_limit=2,
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)
print(f'Steps: {len(train_ds)//BATCH_SIZE*NUM_EPOCHS}, Warmup: {int(len(train_ds)//BATCH_SIZE*NUM_EPOCHS*WARMUP_RATIO)}')
print('Starting training...')
result = trainer.train()
print(f'\nDone! Loss={result.training_loss:.4f}, Time={result.metrics["train_runtime"]/60:.1f}min')

### Plot: Training Curves

In [ ]:
logs = trainer.state.log_history
loss_l = [l for l in logs if 'loss' in l and 'eval_loss' not in l]
eval_l = [l for l in logs if 'eval_f1_macro' in l]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if loss_l:
    steps=[l['step'] for l in loss_l]; losses=[l['loss'] for l in loss_l]
    axes[0].plot(steps, losses, alpha=0.3, color='#2196F3', lw=0.8)
    w=min(10,len(losses)//3)
    if w>1:
        sm=pd.Series(losses).rolling(window=w,center=True).mean()
        axes[0].plot(steps,sm,color='#1565C0',lw=2.5,label=f'Smoothed(w={w})')
    axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss'); axes[0].set_title('Training Loss'); axes[0].legend()

if eval_l:
    ep=[l['epoch'] for l in eval_l]; f1s=[l['eval_f1_macro'] for l in eval_l]; accs=[l['eval_accuracy'] for l in eval_l]
    el=[l.get('eval_loss') for l in eval_l]
    ax2=axes[1].twinx()
    l1=axes[1].plot(ep,f1s,'o-',color='#F44336',lw=2.5,ms=8,label='F1 Macro')
    l2=axes[1].plot(ep,accs,'s--',color='#4CAF50',lw=2,ms=7,label='Accuracy')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1/Acc'); axes[1].set_ylim(0.7,1.0); axes[1].set_title('Validation Metrics')
    vl=[(e,v) for e,v in zip(ep,el) if v is not None]
    if vl:
        ep2,vl2=zip(*vl)
        l3=ax2.plot(ep2,vl2,'^-',color='#FF9800',lw=2,ms=7,label='Eval Loss')
        ax2.set_ylabel('Loss',color='#FF9800'); lines=l1+l2+l3
    else: lines=l1+l2
    axes[1].legend(lines,[l.get_label() for l in lines],loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR,'training_curves.png'),bbox_inches='tight'); plt.show()

## 6. Evaluation

In [ ]:
val_m = trainer.evaluate(val_ds)
print(f'Val  F1={val_m["eval_f1_macro"]:.4f}  Acc={val_m["eval_accuracy"]:.4f}')

test_m = trainer.evaluate(test_ds)
print(f'Test F1={test_m["eval_f1_macro"]:.4f}  Acc={test_m["eval_accuracy"]:.4f}')

tp = trainer.predict(test_ds)
y_pred = np.argmax(tp.predictions, axis=-1)
y_true = tp.label_ids
print('\n' + classification_report(y_true, y_pred, target_names=['Non-Offensive','Offensive']))

### Plot: Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Non-Off','Offensive'], yticklabels=['Non-Off','Offensive'])
axes[0].set_title('Confusion Matrix (Counts)'); axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

cm_n = cm.astype(float)/cm.sum(axis=1)[:,np.newaxis]
sns.heatmap(cm_n, annot=True, fmt='.1%', cmap='YlOrRd', ax=axes[1],
            xticklabels=['Non-Off','Offensive'], yticklabels=['Non-Off','Offensive'], vmin=0, vmax=1)
axes[1].set_title('Confusion Matrix (Normalized)'); axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR,'confusion_matrix.png'),bbox_inches='tight'); plt.show()

tn,fp,fn,tp_v = cm.ravel()
print(f'TN={tn:,} FP={fp:,} FN={fn:,} TP={tp_v:,}')
print(f'FPR={fp/(fp+tn):.3f} (over-censorship)  FNR={fn/(fn+tp_v):.3f} (missed toxicity)')

### Plot: Error Analysis

In [ ]:
tdf = test_df.copy(); tdf['pred']=y_pred; tdf['ok']=y_pred==y_true
fp_df = tdf[(y_true==0)&(y_pred==1)]; fn_df = tdf[(y_true==1)&(y_pred==0)]
print(f'Errors: FP={len(fp_df)} ({len(fp_df)/len(tdf):.1%}), FN={len(fn_df)} ({len(fn_df)/len(tdf):.1%})')
if len(fp_df): print('\nFalse Positives:'); [print(f'  "{r["text"][:100]}"') for _,r in fp_df.head(5).iterrows()]
if len(fn_df): print('\nFalse Negatives:'); [print(f'  "{r["text"][:100]}"') for _,r in fn_df.head(5).iterrows()]

fig,ax=plt.subplots(figsize=(10,4))
tdf['tlen']=tdf['text_norm'].str.len()
ax.hist(tdf[tdf['ok']]['tlen'],bins=40,alpha=0.6,color='#4CAF50',label=f'Correct({tdf["ok"].sum()})',density=True)
ax.hist(tdf[~tdf['ok']]['tlen'],bins=40,alpha=0.6,color='#F44336',label=f'Errors({(~tdf["ok"]).sum()})',density=True)
ax.set_xlabel('Text Length'); ax.set_ylabel('Density'); ax.set_title('Error Distribution by Text Length'); ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(PLOTS_DIR,'error_analysis.png'),bbox_inches='tight'); plt.show()

## 7. Save Model

In [ ]:
SAVE_DIR = os.path.join(OUTPUT_DIR, 'final')
os.makedirs(SAVE_DIR, exist_ok=True)
trainer.save_model(SAVE_DIR); tokenizer.save_pretrained(SAVE_DIR)

metrics = {'model': MODEL_NAME,
    'dataset': {'train': len(train_ds), 'val': len(val_ds), 'test': len(test_ds)},
    'hyperparameters': {'lr': LEARNING_RATE, 'epochs': NUM_EPOCHS, 'batch': BATCH_SIZE, 'max_len': MAX_LENGTH},
    'val': {k: val_m[k] for k in ['eval_f1_macro','eval_f1_weighted','eval_accuracy']},
    'test': {k: test_m[k] for k in ['eval_f1_macro','eval_f1_weighted','eval_accuracy']},
    'training': {'loss': result.training_loss, 'seconds': result.metrics['train_runtime']}}
with open(os.path.join(SAVE_DIR,'eval_metrics.json'),'w') as f: json.dump(metrics,f,indent=2)
print(f'Saved: {SAVE_DIR}')
for fn in sorted(os.listdir(SAVE_DIR)):
    sz=os.path.getsize(os.path.join(SAVE_DIR,fn)) if os.path.isfile(os.path.join(SAVE_DIR,fn)) else 0
    print(f'  {fn} ({sz:,}b)')

## 8. Inference Test

In [ ]:
from transformers import pipeline
clf = pipeline('text-classification', model=SAVE_DIR, tokenizer=SAVE_DIR, device=0 if torch.cuda.is_available() else -1)

msgs = [('\u063A\u0627\u062F\u064A \u0646\u0642\u062A\u0644\u0643 \u064A\u0627 \u0648\u0644\u062F \u0627\u0644\u062D\u0631\u0627\u0645','Threat-OFF'),
        ('\u0646\u062A\u0627 \u062D\u0645\u0627\u0631 \u062F\u064A\u0627\u0644 \u0627\u0644\u0632\u0646\u0642\u0629','Insult-OFF'),
        ('sir t9awed a wld l9a7ba','Sexual insult-OFF'),
        ('nta 7mar ghadi ndarbek','Threat Arabizi-OFF'),
        ('wallah je vais te frapper','Mixed threat-OFF'),
        ('\u0633\u0644\u0627\u0645 \u062E\u0648\u064A\u0627 \u0644\u0627\u0628\u0627\u0633 \u0639\u0644\u064A\u0643\u061F','Safe-NON'),
        ('cv sahbi? fach radi nmchiw?','Safe Arabizi-NON'),
        ('\u0627\u0644\u0644\u0647 \u064A\u0631\u062D\u0645 \u0627\u0644\u0648\u0627\u0644\u062F\u064A\u0646','Gratitude-NON')]

print('=== M1D Inference Test ===')
for t,exp in msgs:
    r=clf(normalize_darija(t))[0]
    lab='OFF' if r['label']=='LABEL_1' else 'NON'
    print(f'  {lab}({r["score"]:.3f}) [{exp}] {t[:70]}')

## 9. Benchmark: XLM-R M1 V3 vs DarijaBERT M1D

In [ ]:
M1_PATH = '/content/drive/MyDrive/AEGIS_ML/models/xlmr-base-m1-v3'
xlmr_f1 = xlmr_acc = None

if os.path.exists(M1_PATH):
    print(f'Benchmarking XLM-R M1 V3 from: {M1_PATH}')
    xp = pipeline('text-classification', model=M1_PATH, tokenizer=M1_PATH,
                  device=0 if torch.cuda.is_available() else -1, batch_size=32)
    xr = xp(test_df['text_norm'].tolist(), truncation=True, max_length=128)
    xp_preds = [1 if r['label']=='LABEL_1' else 0 for r in xr]
    yt = test_df['label'].values
    xlmr_f1 = f1_score(yt, xp_preds, average='macro')
    xlmr_acc = accuracy_score(yt, xp_preds)
    print(f'XLM-R M1: F1={xlmr_f1:.4f} Acc={xlmr_acc:.4f}')
    print(classification_report(yt, xp_preds, target_names=['Non-Off','Offensive']))
else:
    print(f'M1 not found at {M1_PATH}, skipping benchmark')

### Plot: Model Comparison

In [ ]:
if xlmr_f1 is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    models = ['XLM-R M1 V3\n(English)', 'DarijaBERT M1D\n(Darija)']
    f1s = [xlmr_f1, test_m['eval_f1_macro']]; accs = [xlmr_acc, test_m['eval_accuracy']]
    cm2 = ['#FF9800','#4CAF50']
    for ax,vals,title in zip(axes,[f1s,accs],['F1-Macro','Accuracy']):
        bars=ax.bar(models,vals,color=cm2,width=0.5); ax.set_ylim(0.4,1.0); ax.set_ylabel(title); ax.set_title(title+' on Darija Test')
        for b,v in zip(bars,vals): ax.text(b.get_x()+b.get_width()/2,v+0.01,f'{v:.4f}',ha='center',fontsize=13,fontweight='bold')
        d=(vals[1]-vals[0])*100
        ax.annotate(f'+{d:.1f}%',xy=(0.5,0.92),xycoords='axes fraction',fontsize=14,ha='center',color='#2E7D32',fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3',facecolor='#E8F5E9',edgecolor='#4CAF50'))
    plt.suptitle('Darija Offensive Detection: English vs Darija-Native Model',fontsize=14,fontweight='bold',y=1.02)
    plt.tight_layout(); plt.savefig(os.path.join(PLOTS_DIR,'model_comparison.png'),bbox_inches='tight'); plt.show()
else:
    fig,ax=plt.subplots(figsize=(6,5))
    mn=['F1 Macro','F1 Weighted','Accuracy']; mv=[test_m['eval_f1_macro'],test_m['eval_f1_weighted'],test_m['eval_accuracy']]
    bars=ax.bar(mn,mv,color=['#F44336','#2196F3','#4CAF50'],width=0.5); ax.set_ylim(0.5,1.0)
    for b,v in zip(bars,mv): ax.text(b.get_x()+b.get_width()/2,v+0.01,f'{v:.4f}',ha='center',fontsize=13,fontweight='bold')
    ax.set_title('DarijaBERT M1D Test Results')
    plt.tight_layout(); plt.savefig(os.path.join(PLOTS_DIR,'model_comparison.png'),bbox_inches='tight'); plt.show()

## 10. Summary

In [ ]:
fig,ax=plt.subplots(figsize=(12,4)); ax.axis('off')
td=[['Base Model',MODEL_NAME],['Parameters',f'{sum(p.numel() for p in model.parameters()):,}'],
    ['Training Samples',f'{len(train_ds):,}'],['Epochs',str(NUM_EPOCHS)],['LR',str(LEARNING_RATE)],
    ['Training Time',f'{result.metrics["train_runtime"]/60:.1f} min'],['',''],
    ['Val F1-Macro',f'{val_m["eval_f1_macro"]:.4f}'],['Test F1-Macro',f'{test_m["eval_f1_macro"]:.4f}'],
    ['Test Accuracy',f'{test_m["eval_accuracy"]:.4f}']]
if xlmr_f1 is not None:
    td+=[['',''],['XLM-R M1 V3 F1 (Darija)',f'{xlmr_f1:.4f}'],['M1D Improvement',f'+{(test_m["eval_f1_macro"]-xlmr_f1)*100:.1f}%']]
tbl=ax.table(cellText=td,loc='center',cellLoc='left',colWidths=[0.35,0.4])
tbl.auto_set_font_size(False); tbl.set_fontsize(12); tbl.scale(1,1.6)
for i in range(len(td)):
    tbl[i,0].set_facecolor('#E3F2FD'); tbl[i,0].set_text_props(fontweight='bold')
ax.set_title('AEGIS M1D - DarijaBERT-mix Training Summary',fontsize=16,fontweight='bold',pad=20)
plt.savefig(os.path.join(PLOTS_DIR,'summary_table.png'),bbox_inches='tight'); plt.show()

print('\n'+'='*60)
for r in td:
    if r[0]: print(f'  {r[0]:<30} {r[1]}')
print('='*60)
print(f'\nPlots saved to: {PLOTS_DIR}/')
[print(f'  {f}') for f in sorted(os.listdir(PLOTS_DIR))]